In [ ]:
import os
import requests
import json
from mysql import connector
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
load_dotenv(override=True)

# Extraction

In [184]:
API_KEY = os.getenv("API_KEY")
SEASON_ID = 23621

In [185]:
url = f"https://api.sportmonks.com/v3/football/standings/seasons/{SEASON_ID}"
headers = {
	"Content-Type": "application/json"
}
parameters = { "api_token": API_KEY,
    "include": "participant;rule.type;details.type;form;stage;league;group"}


In [186]:
response = requests.get(url, headers=headers , params=parameters)
data = response.json()

In [ ]:
data

# Transformation

In [ ]:
formatted_data = json.dumps(data, indent=4)
print(formatted_data)

In [189]:
def build_standing_df(data):
    rows = []
    for club in data:
        item = {
            "Position": club["position"],
            "Equipe": club["participant"]["name"],
            "Points": club["points"],
        }

        form_data = club.get("form", [])
        form_sorted = sorted(form_data,                  # On trie pour que le match 1 soit avant le match 38
                             key=lambda x:                 # sorted c'est la fonction de base de Python pour trier
                             x.get("sort_order", 0))    
        letters = [f.get("form") for f in form_sorted]  # On ne garde que les lettres (W, D, L)
        item["Form"] = " ".join(letters[-5:])           # On colle les 5 derniers : "WWWDL"


        for detail in club.get("details", []):
            raw_name = detail.get("type", {}).get("name").replace(" ", "_")
            if raw_name:
                stat_name = raw_name.replace(" ", "_")
                item[stat_name] = detail.get("value")


        rows.append(item)
    
    df = pd.DataFrame(rows)
    target_cols = ["Position", "Equipe", "Overall_Matches_Played", "Overall_Won", 
                    "Overall_Draw", "Overall_Lost", "Overal_Goals_Scored", 
                    "Overall_Goals_Conceded", "Goal_Difference", "Form", "Points"]

    cols_presentes = [c for c in target_cols if c in df.columns]
    cols_manquantes = set(target_cols) - set(df.columns)
    if cols_manquantes:
        print(f"⚠️ Attention, colonnes introuvables dans le JSON : {cols_manquantes}")

    return df[cols_presentes].sort_values("Position")

In [190]:
df_classement = build_standing_df(data["data"])
print(df_classement.to_string(index=False))

 Position           Equipe  Overall_Matches_Played  Overall_Won  Overall_Draw  Overall_Lost  Overal_Goals_Scored  Overall_Goals_Conceded  Goal_Difference      Form  Points
        1     FC Barcelona                      38           28             4             6                  102                      39               63 W W W L W      88
        2      Real Madrid                      38           26             6             6                   78                      38               40 W L W W W      84
        3  Atlético Madrid                      38           22            10             6                   68                      30               38 D W L W W      76
        4    Athletic Club                      38           19            13             6                   54                      29               25 D W W W L      70
        5       Villarreal                      38           20            10             8                   71                      51    

In [ ]:
df_classement.info()

# Chargement

In [192]:
MySQL_HOST = os.getenv("MySQL_HOST")
MySQL_PORT = os.getenv("MySQL_PORT")
MySQL_USER = os.getenv("MySQL_USER")
MySQL_PASSWORD = os.getenv("MySQL_PASSWORD")
MySQL_DATABASE = os.getenv("MySQL_DATABASE")

In [193]:
server_config = connector.connect(
    host=MySQL_HOST,
    port=MySQL_PORT,
    user=MySQL_USER,
    password=MySQL_PASSWORD,
    database=MySQL_DATABASE,
    connection_timeout=10
)
cursor = server_config.cursor()
print ("Connexion à la base de données réussie !")

Connexion à la base de données réussie !


In [194]:
# Validation de la connexion
sql_table ="laliga_2024_2025_dw"
cursor.execute(f"SELECT * FROM {sql_table} LIMIT 5;")
result = cursor.fetchall()
print(f"[SUCCESS] Connexion validée, avec la table {sql_table}")


[SUCCESS] Connexion validée, avec la table laliga_2024_2025_dw


In [195]:
table_cols = ["Position", "Equipe", "Overall_Matches_Played", "Overall_Won", 
                    "Overall_Draw", "Overall_Lost", "Overal_Goals_Scored", 
                    "Overall_Goals_Conceded", "Goal_Difference", "Form", "Points"]

liga2024_2025 = df_classement[table_cols]

In [196]:
liga2024_2025_tuples = liga2024_2025.itertuples(index=False, name=None)

list_of_tuples = list(liga2024_2025_tuples)
print(list_of_tuples)

[(1, 'FC Barcelona', 38, 28, 4, 6, 102, 39, 63, 'W W W L W', 88), (2, 'Real Madrid', 38, 26, 6, 6, 78, 38, 40, 'W L W W W', 84), (3, 'Atlético Madrid', 38, 22, 10, 6, 68, 30, 38, 'D W L W W', 76), (4, 'Athletic Club', 38, 19, 13, 6, 54, 29, 25, 'D W W W L', 70), (5, 'Villarreal', 38, 20, 10, 8, 71, 51, 20, 'W W W W W', 70), (6, 'Real Betis', 38, 16, 12, 10, 57, 50, 7, 'W D D L D', 60), (7, 'Celta de Vigo', 38, 16, 7, 15, 59, 57, 2, 'L W W L W', 55), (8, 'Osasuna', 38, 12, 16, 10, 48, 52, -4, 'L D W W D', 52), (9, 'Rayo Vallecano', 38, 13, 13, 12, 41, 45, -4, 'W W D W D', 52), (10, 'Mallorca', 38, 13, 9, 16, 35, 44, -9, 'L W L L D', 48), (11, 'Valencia', 38, 11, 13, 14, 44, 54, -10, 'W W L L D', 46), (12, 'Real Sociedad', 38, 13, 7, 18, 35, 46, -11, 'D L L W L', 46), (13, 'Getafe', 38, 11, 9, 18, 34, 39, -5, 'L L L W L', 42), (14, 'Deportivo Alavés', 38, 10, 12, 16, 38, 48, -10, 'D L W W D', 42), (15, 'Espanyol', 38, 11, 9, 18, 40, 51, -11, 'L L L L W', 42), (16, 'Sevilla', 38, 10, 11, 

In [197]:
UPSERT_SQL = f"""
INSERT INTO {sql_table} (Position, Equipe, Overall_Matches_Played, Overall_Won, Overall_Draw, Overall_Lost, Overal_Goals_Scored, Overall_Goals_Conceded, Goal_Difference, Form, Points)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
Position = VALUES(Position),
Equipe = VALUES(Equipe),        
Overall_Matches_Played = VALUES(Overall_Matches_Played),
Overall_Won = VALUES(Overall_Won),
Overall_Draw = VALUES(Overall_Draw),    
Overall_Lost = VALUES(Overall_Lost),
Overal_Goals_Scored = VALUES(Overal_Goals_Scored),
Overall_Goals_Conceded = VALUES(Overall_Goals_Conceded),
Goal_Difference = VALUES(Goal_Difference),      
Form = VALUES(Form),
Points = VALUES(Points);
"""

In [198]:
nbre_de_lignes_chargee = len(list_of_tuples)

In [ ]:
try:    
    cursor.executemany(UPSERT_SQL, list_of_tuples)
    server_config.commit()   
    print(f"[SUCCESS] {nbre_de_lignes_chargee} lignes insérées ou mises à jour dans la table {sql_table}")
except Exception as e:
    print(f"[ERROR] Une erreur est survenue lors de l'insertion : {e}") 
finally:
    cursor.close()
    server_config.close()   